# Fine-tuning con Hard Negatives - Versión Simplificada

🎯 **Objetivo:** Reducir falsos positivos de `knife` (confusión con celular de costado)

📋 **Requisitos previos:**
1. Subir `hard_negatives_celulares.zip` a Drive (carpeta `procesamiento-imagenes`)
2. Tener checkpoint del modelo actual en Drive

⚙️ **Estrategia:**
- Agregar 28 celulares como hard negatives al training
- Fine-tuning con LR bajo (1e-5) para no destruir conocimiento
- Pocas épocas (10-15) con early stopping

In [ ]:
# Instalar dependencias
!pip install -q torchmetrics

In [ ]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ir al repositorio
%cd /content/drive/MyDrive/procesamiento-imagenes

In [ ]:
# Copiar dataset aumentado a disco local (GPU)
print("📦 Copiando dataset aumentado...")
!cp dataset_augmented.zip /content/
!unzip -q /content/dataset_augmented.zip -d /content/
print("✅ Dataset listo")

In [ ]:
# Copiar hard negatives a disco local
print("📱 Copiando hard negatives (celulares)...")
!cp hard_negatives_celulares.zip /content/
!unzip -q /content/hard_negatives_celulares.zip -d /content/
print("✅ Hard negatives listos")

In [ ]:
# Verificar archivos
print("Imágenes de celulares:")
!ls /content/hard_negatives/images | wc -l
print("\nXMLs de celulares:")
!ls /content/hard_negatives/xmls | wc -l
print("\nDeben ser 28 ambos")

In [ ]:
# Integrar hard negatives al training (NO al test)
print("🔗 Integrando hard negatives al training...")
!cp /content/hard_negatives/images/* /content/dataset_augmented/images/
!cp /content/hard_negatives/xmls/* /content/dataset_augmented/xmls/
print("✅ Integración completa")

print("\nTotal training ahora:")
!ls /content/dataset_augmented/images | wc -l

## Fine-tuning

⚠️ **IMPORTANTE:** Ajustá la ruta del checkpoint según tu modelo actual.

Hiperparámetros para fine-tuning:
- `--lr 1e-5`: Learning rate bajo para no destruir conocimiento
- `--epochs 15`: Pocas épocas
- `--batch-size 6`: Batch pequeño para convergencia suave
- `--patience 5`: Early stopping agresivo

In [ ]:
# OPCIÓN 1: Fine-tuning desde checkpoint anterior
# Ajustá la ruta del checkpoint
!python3 pipeline_entrenamiento.py \
  --skip-stages split augment \
  --resume /content/drive/MyDrive/procesamiento-imagenes/results_standard/best_model.pth \
  --augmented-images /content/dataset_augmented/images \
  --augmented-xmls /content/dataset_augmented/xmls \
  --epochs 15 \
  --batch-size 6 \
  --lr 1e-5 \
  --enhance \
  --amp \
  --save-every 5 \
  --patience 5 \
  --output-dir results_finetuning_negatives

In [ ]:
# OPCIÓN 2: Entrenar desde cero (si no tenés checkpoint)
# !python3 pipeline_entrenamiento.py \
#   --skip-stages split \
#   --augmented-images /content/dataset_augmented/images \
#   --augmented-xmls /content/dataset_augmented/xmls \
#   --epochs 50 \
#   --batch-size 8 \
#   --num-augmentations 3 \
#   --enhance \
#   --amp \
#   --save-every 10 \
#   --patience 5 \
#   --output-dir results_with_negatives

## Evaluación

Métricas clave a monitorear:
1. **FP de knife:** Debe bajar (objetivo principal)
2. **Recall de knife:** No debe bajar más de 2-3%
3. **Precision de knife:** Debe subir
4. **mAP general:** Verificar que no empeore

In [ ]:
# Ver resultados
!ls -lh results_finetuning_negatives/
print("\n📊 Métricas:")
!cat results_finetuning_negatives/training_log.txt | tail -20

In [ ]:
# Copiar resultados a Drive
!cp -r results_finetuning_negatives /content/drive/MyDrive/procesamiento-imagenes/
print("✅ Resultados guardados en Drive")

## Comparación Antes/Después

Para verificar la mejora, compará:
- Modelo anterior: FP de knife en test
- Modelo nuevo: FP de knife en test (debe ser menor)
- Recall de knife (no debe bajar significativamente)

In [ ]:
# TODO: Agregar comparación de métricas
# Podés usar el script de testing para comparar ambos modelos